
# Fold-Level Statistical Analysis for HAM10000 Attention Experiments

This notebook **does not train any model**. It recursively reads completed experiment outputs, extracts the five original fold-level macro-F1 values for E1–E10, validates data quality and manuscript summaries, verifies paired validation splits when identifiers are available, performs eight paired comparisons, applies Holm correction, and writes:

- `fold_level_macro_f1.csv`
- `baseline_statistical_comparison.csv`
- `baseline_statistical_comparison.tex`
- `statistical_analysis_report.md`

The notebook is intentionally strict. It stops with an informative error if an experiment does not contain exactly five unique folds, contains invalid values, or fails the manuscript sanity check.


In [1]:

# =========================
# 1. CONFIGURATION
# =========================
from pathlib import Path

ROOT_DIR = Path("/Users/asfanme/RISET/Riset4")
OUTPUT_DIR = ROOT_DIR / "statistical_analysis_outputs"

# Folder mapping based on the supplied project structure.
EXPERIMENT_DIRS = {
    "E1": ROOT_DIR / "results_exp0",
    "E2": ROOT_DIR / "results_exp1",
    "E3": ROOT_DIR / "results_exp2",
    "E4": ROOT_DIR / "results_exp3",
    "E5": ROOT_DIR / "results_exp4",
    "E6": ROOT_DIR / "results_exp0_group",
    "E7": ROOT_DIR / "results_exp1_group",
    "E8": ROOT_DIR / "results_exp2_group",
    "E9": ROOT_DIR / "results_exp3_group",
    "E10": ROOT_DIR / "results_exp4_group",
}

EXPERIMENT_META = {
    "E1":  {"protocol": "Image-level Stratified 5-Fold", "attention": "Baseline"},
    "E2":  {"protocol": "Image-level Stratified 5-Fold", "attention": "Channel Attention"},
    "E3":  {"protocol": "Image-level Stratified 5-Fold", "attention": "Spatial Attention"},
    "E4":  {"protocol": "Image-level Stratified 5-Fold", "attention": "CBAM"},
    "E5":  {"protocol": "Image-level Stratified 5-Fold", "attention": "Coordinate Attention"},
    "E6":  {"protocol": "Lesion-level Group 5-Fold", "attention": "Baseline"},
    "E7":  {"protocol": "Lesion-level Group 5-Fold", "attention": "Channel Attention"},
    "E8":  {"protocol": "Lesion-level Group 5-Fold", "attention": "Spatial Attention"},
    "E9":  {"protocol": "Lesion-level Group 5-Fold", "attention": "CBAM"},
    "E10": {"protocol": "Lesion-level Group 5-Fold", "attention": "Coordinate Attention"},
}

COMPARISONS = [
    ("E2", "E1"), ("E3", "E1"), ("E4", "E1"), ("E5", "E1"),
    ("E7", "E6"), ("E8", "E6"), ("E9", "E6"), ("E10", "E6"),
]

MANUSCRIPT_SUMMARY = {
    "E1":  (83.42, 1.68),
    "E2":  (84.68, 1.53),
    "E3":  (83.06, 2.33),
    "E4":  (82.36, 3.20),
    "E5":  (81.90, 2.13),
    "E6":  (72.58, 2.93),
    "E7":  (70.51, 3.14),
    "E8":  (73.96, 3.37),
    "E9":  (72.23, 3.89),
    "E10": (72.02, 2.87),
}

# Tolerances are in percentage points. They accommodate display rounding only.
SANITY_MEAN_TOLERANCE_PP = 0.08
SANITY_SD_TOLERANCE_PP = 0.12

# Accepted source files. Add extensions here when necessary.
TABULAR_EXTENSIONS = {
    ".csv", ".tsv", ".txt", ".json", ".jsonl", ".xlsx", ".xls",
    ".parquet", ".feather", ".pkl", ".pickle"
}
ID_FILE_EXTENSIONS = TABULAR_EXTENSIONS | {".npy", ".npz"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR   : {ROOT_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


ROOT_DIR   : /Users/asfanme/RISET/Riset4
OUTPUT_DIR : /Users/asfanme/RISET/Riset4/statistical_analysis_outputs


In [2]:

# =========================
# 2. IMPORTS AND UTILITIES
# =========================
import json
import math
import re
import warnings
from collections import defaultdict
from itertools import product
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

FOLD_PATTERNS = [
    re.compile(r"(?:^|[_\-\s/])fold[_\-\s]*(\d+)(?:$|[_\-\s/.])", re.I),
    re.compile(r"(?:^|[_\-\s/])f[_\-\s]*(\d+)(?:$|[_\-\s/.])", re.I),
]

MACRO_F1_ALIASES = {
    "macrof1", "macrof1score", "f1macro", "f1scoremacro",
    "valmacrof1", "validationmacrof1", "bestvalmacrof1",
    "testmacrof1", "macroaveragef1", "macroavgf1"
}

F1_COLUMN_ALIASES = {
    "f1score", "f1", "f1scores"
}

FOLD_ALIASES = {
    "fold", "foldid", "foldnumber", "foldnum", "cvfold", "split", "splitid"
}

MACRO_ROW_ALIASES = {
    "macroavg", "macroaverage", "macro", "macroaveragef1", "macroavgf1"
}

def normalize_name(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

def parse_fold_from_text(text: str) -> Optional[int]:
    text = str(text)
    for pattern in FOLD_PATTERNS:
        match = pattern.search(text)
        if match:
            return int(match.group(1))
    match = re.search(r"fold\s*[_\-]?\s*(\d+)", text, re.I)
    return int(match.group(1)) if match else None

def read_tabular_file(path: Path) -> List[pd.DataFrame]:
    """Read a supported file into one or more DataFrames."""
    suffix = path.suffix.lower()
    frames: List[pd.DataFrame] = []

    try:
        if suffix == ".csv":
            # Standard sklearn classification_report CSV often stores row labels
            # in the first unnamed column. Read normally; the extractor handles it.
            frames.append(pd.read_csv(path))
        elif suffix == ".tsv":
            frames.append(pd.read_csv(path, sep="\t"))
        elif suffix == ".txt":
            loaded = False
            for sep in [None, "\t", ",", ";", r"\s+"]:
                try:
                    df = pd.read_csv(path, sep=sep, engine="python")
                    if not df.empty and len(df.columns) >= 1:
                        frames.append(df)
                        loaded = True
                        break
                except Exception:
                    pass
            if not loaded:
                text = path.read_text(encoding="utf-8", errors="ignore")
                records = []
                for line in text.splitlines():
                    m = re.search(
                        r"(?:macro[\s_\-]*f1|f1[\s_\-]*macro)\s*[:=]\s*"
                        r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line, re.I
                    )
                    if m:
                        records.append({"macro_f1": float(m.group(1))})
                if records:
                    frames.append(pd.DataFrame(records))
        elif suffix in {".xlsx", ".xls"}:
            book = pd.ExcelFile(path)
            for sheet in book.sheet_names:
                frames.append(pd.read_excel(path, sheet_name=sheet))
        elif suffix == ".json":
            obj = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
            if isinstance(obj, list):
                frames.append(pd.json_normalize(obj))
            elif isinstance(obj, dict):
                frames.append(pd.json_normalize(obj))
                for _, val in obj.items():
                    if isinstance(val, list):
                        try:
                            frames.append(pd.json_normalize(val))
                        except Exception:
                            pass
                    elif isinstance(val, dict):
                        try:
                            frames.append(pd.json_normalize(val))
                        except Exception:
                            pass
        elif suffix == ".jsonl":
            frames.append(pd.read_json(path, lines=True))
        elif suffix == ".parquet":
            frames.append(pd.read_parquet(path))
        elif suffix == ".feather":
            frames.append(pd.read_feather(path))
        elif suffix in {".pkl", ".pickle"}:
            obj = pd.read_pickle(path)
            if isinstance(obj, pd.DataFrame):
                frames.append(obj)
            elif isinstance(obj, (dict, list)):
                frames.append(pd.json_normalize(obj))
    except Exception as exc:
        raise RuntimeError(f"Failed to read {path}: {exc}") from exc

    return [df for df in frames if isinstance(df, pd.DataFrame) and not df.empty]

def candidate_macro_columns(df: pd.DataFrame) -> List[str]:
    candidates = []
    for col in df.columns:
        norm = normalize_name(col)
        if norm in MACRO_F1_ALIASES:
            candidates.append(col)
        elif "macro" in norm and "f1" in norm:
            if not any(token in norm for token in ["mean", "std", "sd", "ci", "lower", "upper"]):
                candidates.append(col)
    return candidates

def candidate_f1_columns(df: pd.DataFrame) -> List[str]:
    return [col for col in df.columns if normalize_name(col) in F1_COLUMN_ALIASES]

def candidate_fold_columns(df: pd.DataFrame) -> List[str]:
    return [col for col in df.columns if normalize_name(col) in FOLD_ALIASES]

def numeric_series(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.replace("%", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")

def normalize_metric_scale(values: Sequence[float]) -> Tuple[np.ndarray, str]:
    arr = np.asarray(values, dtype=float)
    if np.any(~np.isfinite(arr)):
        raise ValueError("Macro-F1 contains non-finite values.")
    if np.all((arr >= 0) & (arr <= 1.000001)):
        return arr * 100.0, "0–1 converted to percent"
    if np.all((arr >= 0) & (arr <= 100.000001)):
        return arr, "already percent"
    raise ValueError(f"Macro-F1 values outside plausible 0–1 or 0–100 range: {arr.tolist()}")

def extract_sklearn_classification_report(
    df: pd.DataFrame, path: Path, frame_idx: int
) -> List[Dict[str, Any]]:
    """
    Extract the 'macro avg' row from a sklearn classification_report CSV.

    Typical layout:
        Unnamed: 0, precision, recall, f1-score, support
        ...
        macro avg, 0.xxx, 0.xxx, 0.xxx, n
    """
    rows: List[Dict[str, Any]] = []
    f1_cols = candidate_f1_columns(df)
    if not f1_cols:
        return rows

    file_fold = parse_fold_from_text(str(path))
    if file_fold is None:
        return rows

    # Search every non-F1 column and also the index for the 'macro avg' label.
    label_sources = [(str(col), df[col].astype(str)) for col in df.columns if col not in f1_cols]
    label_sources.append(("__index__", pd.Series(df.index.astype(str), index=df.index)))

    for f1_col in f1_cols:
        f1_values = numeric_series(df[f1_col])
        for label_col, labels in label_sources:
            normalized_labels = labels.map(normalize_name)
            mask = normalized_labels.isin(MACRO_ROW_ALIASES)
            for row_idx in df.index[mask]:
                value = f1_values.loc[row_idx]
                if pd.notna(value):
                    rows.append({
                        "fold_raw": int(file_fold),
                        "macro_f1_raw": float(value),
                        "source_file": str(path),
                        "frame_index": frame_idx,
                        "row_index": int(row_idx) if isinstance(row_idx, (int, np.integer)) else str(row_idx),
                        "macro_column": f"{f1_col} @ row '{labels.loc[row_idx]}'",
                    })
    return rows

def extract_candidates_from_file(path: Path) -> List[Dict[str, Any]]:
    """
    Extract fold-level macro-F1 candidates.

    Supported layouts include:
    1. A direct macro_f1 / val_macro_f1 column.
    2. sklearn classification_report CSV:
       row='macro avg', column='f1-score'.
    """
    candidates: List[Dict[str, Any]] = []
    file_fold = parse_fold_from_text(str(path))
    frames = read_tabular_file(path)

    for frame_idx, df in enumerate(frames):
        # First: explicit sklearn classification-report layout.
        candidates.extend(extract_sklearn_classification_report(df, path, frame_idx))

        # Second: direct macro-F1 columns.
        macro_cols = candidate_macro_columns(df)
        fold_cols = candidate_fold_columns(df)

        for macro_col in macro_cols:
            values = numeric_series(df[macro_col])
            for row_idx, value in values.items():
                if pd.isna(value):
                    continue

                row_text = " ".join(str(x) for x in df.loc[row_idx].tolist()).lower()
                if re.search(r"\b(mean|average|avg|std|standard deviation|summary|overall)\b", row_text):
                    explicit_fold_present = False
                    for fc in fold_cols:
                        try:
                            explicit_fold_present |= pd.notna(df.loc[row_idx, fc])
                        except Exception:
                            pass
                    if not explicit_fold_present:
                        continue

                fold = None
                for fold_col in fold_cols:
                    raw_fold = df.loc[row_idx, fold_col]
                    if pd.notna(raw_fold):
                        try:
                            fold = int(float(raw_fold))
                            break
                        except Exception:
                            parsed = parse_fold_from_text(str(raw_fold))
                            if parsed is not None:
                                fold = parsed
                                break

                if fold is None:
                    fold = parse_fold_from_text(row_text)
                if fold is None:
                    fold = file_fold

                if fold is not None:
                    candidates.append({
                        "fold_raw": int(fold),
                        "macro_f1_raw": float(value),
                        "source_file": str(path),
                        "frame_index": frame_idx,
                        "row_index": int(row_idx) if isinstance(row_idx, (int, np.integer)) else str(row_idx),
                        "macro_column": str(macro_col),
                    })
    return candidates

def canonicalize_fold_numbers(folds: Sequence[int]) -> Dict[int, int]:
    unique = sorted(set(int(x) for x in folds))
    if unique == [0, 1, 2, 3, 4]:
        return {x: x + 1 for x in unique}
    if unique == [1, 2, 3, 4, 5]:
        return {x: x for x in unique}
    raise ValueError(
        f"Expected fold labels 0–4 or 1–5, but found {unique}. "
        "Check files for duplicate, missing, or non-fold summary rows."
    )

def discover_experiment_macro_f1(experiment: str, folder: Path) -> Tuple[pd.DataFrame, List[Path], str]:
    if not folder.exists():
        raise FileNotFoundError(f"{experiment}: experiment folder does not exist: {folder}")

    files = sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in TABULAR_EXTENSIONS
    )
    if not files:
        raise FileNotFoundError(f"{experiment}: no supported result files found under {folder}")

    all_candidates: List[Dict[str, Any]] = []
    parsed_files: List[Path] = []
    errors: List[str] = []

    for path in files:
        try:
            rows = extract_candidates_from_file(path)
            if rows:
                all_candidates.extend(rows)
                parsed_files.append(path)
        except Exception as exc:
            errors.append(f"{path}: {exc}")

    if not all_candidates:
        error_note = "\n".join(errors[:20])
        raise RuntimeError(
            f"{experiment}: no fold-level macro-F1 candidates were found.\n"
            f"Files inspected: {len(files)}\nRead errors:\n{error_note}\n"
            "Expected either a direct macro-F1 column or a classification-report "
            "row named 'macro avg' with an 'f1-score' column."
        )

    cand = pd.DataFrame(all_candidates)

    groups = []
    for _, group in cand.groupby(["source_file", "frame_index", "macro_column"], dropna=False):
        try:
            fmap = canonicalize_fold_numbers(group["fold_raw"].tolist())
        except ValueError:
            continue
        tmp = group.copy()
        tmp["fold"] = tmp["fold_raw"].map(fmap)
        if tmp["fold"].nunique() == 5 and not tmp["fold"].duplicated().any():
            groups.append(tmp)

    selected = None
    selection_reason = ""

    if groups:
        groups.sort(key=lambda g: (
            len(g) != 5,
            "summary" in str(g["source_file"].iloc[0]).lower(),
            len(str(g["source_file"].iloc[0]))
        ))
        selected = groups[0].copy()
        selection_reason = "complete five-fold table"
    else:
        # Classification reports normally provide one file per fold.
        unique_rows = cand.drop_duplicates(subset=["fold_raw", "macro_f1_raw"]).copy()
        try:
            fmap = canonicalize_fold_numbers(unique_rows["fold_raw"].tolist())
            unique_rows["fold"] = unique_rows["fold_raw"].map(fmap)
        except ValueError as exc:
            diagnostic = cand.sort_values(["fold_raw", "source_file"]).to_string(index=False)
            raise RuntimeError(
                f"{experiment}: unable to form a unique five-fold vector. {exc}\n"
                f"Candidate rows:\n{diagnostic}"
            ) from exc

        chosen_rows = []
        for fold in range(1, 6):
            g = unique_rows[unique_rows["fold"] == fold]
            if g.empty:
                raise RuntimeError(f"{experiment}: missing fold {fold}.")
            vals = g["macro_f1_raw"].to_numpy(float)
            vals_pp = np.where(vals <= 1.000001, vals * 100.0, vals)
            if np.ptp(vals_pp) > 1e-6:
                raise RuntimeError(
                    f"{experiment}: conflicting macro-F1 values for fold {fold}:\n"
                    + g.to_string(index=False)
                )
            chosen_rows.append(g.iloc[0])
        selected = pd.DataFrame(chosen_rows)
        selection_reason = "one macro-avg F1 value assembled from each fold file"

    selected = selected.sort_values("fold").reset_index(drop=True)
    scaled, scale_note = normalize_metric_scale(selected["macro_f1_raw"].to_numpy(float))
    selected["macro_f1"] = scaled

    if len(selected) != 5 or selected["fold"].nunique() != 5:
        raise RuntimeError(f"{experiment}: extraction did not yield exactly five unique folds.")
    if selected["macro_f1"].isna().any():
        raise RuntimeError(f"{experiment}: macro-F1 contains missing values.")
    if selected["fold"].tolist() != [1, 2, 3, 4, 5]:
        raise RuntimeError(f"{experiment}: final fold ordering is invalid: {selected['fold'].tolist()}")
    if not selected["macro_f1"].between(0, 100).all():
        raise RuntimeError(f"{experiment}: macro-F1 outside 0–100 after scale normalization.")

    selected.insert(0, "experiment", experiment)
    selected["protocol"] = EXPERIMENT_META[experiment]["protocol"]
    selected["attention"] = EXPERIMENT_META[experiment]["attention"]

    final_cols = [
        "experiment", "protocol", "attention", "fold", "macro_f1",
        "source_file", "macro_column"
    ]
    return selected[final_cols], sorted(set(parsed_files)), f"{selection_reason}; {scale_note}"


In [3]:

# Optional quick diagnostic: inspect one classification-report file.
example_report = EXPERIMENT_DIRS["E1"] / "exp0_fold1_classification_report.csv"
if example_report.exists():
    preview = pd.read_csv(example_report)
    print(f"Preview: {example_report}")
    display(preview)
    print("Detected F1 columns:", candidate_f1_columns(preview))
else:
    print("Example report not found at the expected path; recursive discovery will still be used.")


Preview: /Users/asfanme/RISET/Riset4/results_exp0/exp0_fold1_classification_report.csv


,Unnamed: 0,precision,recall,f1-score,support
0,akiec,0.810345,0.723077,0.764228,65.000000
1,bcc,0.798319,0.922330,0.855856,103.000000
2,bkl,0.843902,0.786364,0.814118,220.000000
3,df,1.000000,0.782609,0.878049,23.000000
4,mel,0.813830,0.686099,0.744526,223.000000
5,nv,0.938584,0.968680,0.953394,1341.000000
6,vasc,0.838710,0.928571,0.881356,28.000000
7,accuracy,0.904144,0.904144,0.904144,0.904144
8,macro avg,0.863384,0.828247,0.841647,2003.000000
9,weighted avg,0.902230,0.904144,0.901816,2003.000000


Detected F1 columns: ['f1-score']


In [4]:

# =========================
# 3. EXTRACT ALL FOLD VALUES
# =========================
extracted_frames = []
discovery_log = {}
fatal_errors = []

for experiment in [f"E{i}" for i in range(1, 11)]:
    folder = EXPERIMENT_DIRS[experiment]
    try:
        df_exp, found_files, note = discover_experiment_macro_f1(experiment, folder)
        extracted_frames.append(df_exp)
        discovery_log[experiment] = {
            "folder": str(folder),
            "files": [str(p) for p in found_files],
            "note": note,
        }
    except Exception as exc:
        fatal_errors.append(f"{experiment}: {exc}")

if fatal_errors:
    raise RuntimeError(
        "Extraction stopped because one or more experiments failed validation:\n\n"
        + "\n\n".join(fatal_errors)
    )

fold_df = pd.concat(extracted_frames, ignore_index=True)
fold_df = fold_df.sort_values(["experiment", "fold"], key=lambda s: s.map(
    {f"E{i}": i for i in range(1, 11)}
) if s.name == "experiment" else s).reset_index(drop=True)

print("FILES USED FOR EACH EXPERIMENT")
print("=" * 80)
for experiment in [f"E{i}" for i in range(1, 11)]:
    print(f"\n{experiment} — {discovery_log[experiment]['note']}")
    for file in discovery_log[experiment]["files"]:
        print(f"  - {file}")

print("\n\nFIVE EXTRACTED MACRO-F1 VALUES (%)")
print("=" * 80)
display(
    fold_df.pivot(index="experiment", columns="fold", values="macro_f1")
           .rename(columns=lambda x: f"Fold {x}")
           .round(4)
)


FILES USED FOR EACH EXPERIMENT

E1 — one macro-avg F1 value assembled from each fold file; 0–1 converted to percent
  - /Users/asfanme/RISET/Riset4/results_exp0/exp0_fold1_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp0/exp0_fold2_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp0/exp0_fold3_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp0/exp0_fold4_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp0/exp0_fold5_classification_report.csv

E2 — one macro-avg F1 value assembled from each fold file; 0–1 converted to percent
  - /Users/asfanme/RISET/Riset4/results_exp1/exp1_fold1_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp1/exp1_fold2_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp1/exp1_fold3_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp1/exp1_fold4_classification_report.csv
  - /Users/asfanme/RISET/Riset4/results_exp1/exp1_fold5_class

fold,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5
experiment,,,,,
E1,84.1647,84.3225,81.7251,85.3227,81.5562
E10,68.7325,74.0166,72.2990,75.4772,69.5553
E2,85.9697,85.3918,82.1112,85.4198,84.4876
E3,84.3887,83.9058,78.9432,83.6491,84.4271
E4,87.5793,81.4202,82.2699,81.6934,78.8393
E5,83.4508,84.6373,79.4288,81.5202,80.4566
E6,70.7400,74.7553,72.1678,69.0094,76.2334
E7,65.4734,73.2610,73.0220,70.5863,70.2235
E8,69.6588,79.0099,74.0434,74.3133,72.7680


In [5]:

# =========================
# 4. DATA QUALITY AND MANUSCRIPT SANITY CHECK
# =========================
quality_rows = []
sanity_failures = []

for experiment, group in fold_df.groupby("experiment", sort=False):
    values = group.sort_values("fold")["macro_f1"].to_numpy(float)
    extracted_mean = values.mean()
    extracted_sd = values.std(ddof=1)
    manuscript_mean, manuscript_sd = MANUSCRIPT_SUMMARY[experiment]

    checks = {
        "exactly_five_folds": len(group) == 5,
        "unique_folds": group["fold"].nunique() == 5,
        "no_missing_values": not group["macro_f1"].isna().any(),
        "plausible_range": group["macro_f1"].between(0, 100).all(),
        "mean_matches": abs(extracted_mean - manuscript_mean) <= SANITY_MEAN_TOLERANCE_PP,
        "sd_matches": abs(extracted_sd - manuscript_sd) <= SANITY_SD_TOLERANCE_PP,
    }
    passed = all(checks.values())

    quality_rows.append({
        "experiment": experiment,
        "n_folds": len(group),
        "extracted_mean": extracted_mean,
        "manuscript_mean": manuscript_mean,
        "mean_abs_difference": abs(extracted_mean - manuscript_mean),
        "extracted_sd": extracted_sd,
        "manuscript_sd": manuscript_sd,
        "sd_abs_difference": abs(extracted_sd - manuscript_sd),
        "status": "PASS" if passed else "FAIL",
        **checks,
    })

    if not passed:
        failed_items = [k for k, v in checks.items() if not v]
        sanity_failures.append(
            f"{experiment}: failed {failed_items}; values={values.tolist()}, "
            f"extracted={extracted_mean:.4f} ± {extracted_sd:.4f}, "
            f"manuscript={manuscript_mean:.2f} ± {manuscript_sd:.2f}"
        )

quality_df = pd.DataFrame(quality_rows)
display(quality_df.round(4))

if sanity_failures:
    raise RuntimeError(
        "Sanity check failed. Statistical testing has been stopped; no original "
        "fold values were altered.\n\n" + "\n".join(sanity_failures)
    )

print("All ten experiments passed the five-fold, missingness, uniqueness, range, and manuscript sanity checks.")


,experiment,n_folds,extracted_mean,manuscript_mean,mean_abs_difference,extracted_sd,manuscript_sd,sd_abs_difference,status,exactly_five_folds,unique_folds,no_missing_values,plausible_range,mean_matches,sd_matches
0,E1,5,83.4182,83.42,0.0018,1.6834,1.68,0.0034,PASS,True,True,True,True,True,True
1,E2,5,84.6760,84.68,0.0040,1.5291,1.53,0.0009,PASS,True,True,True,True,True,True
2,E3,5,83.0628,83.06,0.0028,2.3262,2.33,0.0038,PASS,True,True,True,True,True,True
3,E4,5,82.3604,82.36,0.0004,3.2005,3.20,0.0005,PASS,True,True,True,True,True,True
4,E5,5,81.8987,81.90,0.0013,2.1349,2.13,0.0049,PASS,True,True,True,True,True,True
5,E6,5,72.5812,72.58,0.0012,2.9319,2.93,0.0019,PASS,True,True,True,True,True,True
6,E7,5,70.5132,70.51,0.0032,3.1358,3.14,0.0042,PASS,True,True,True,True,True,True
7,E8,5,73.9587,73.96,0.0013,3.3747,3.37,0.0047,PASS,True,True,True,True,True,True
8,E9,5,72.2320,72.23,0.0020,3.8906,3.89,0.0006,PASS,True,True,True,True,True,True
9,E10,5,72.0161,72.02,0.0039,2.8679,2.87,0.0021,PASS,True,True,True,True,True,True


All ten experiments passed the five-fold, missingness, uniqueness, range, and manuscript sanity checks.


In [6]:

# =========================
# 5. OPTIONAL VALIDATION-SPLIT ID VERIFICATION
# =========================
ID_NAME_HINTS = (
    "val", "valid", "validation", "split", "fold",
    "sample", "image", "lesion", "index", "indices", "ids"
)
ID_COLUMN_HINTS = (
    "sampleid", "imageid", "lesionid", "id", "index", "idx",
    "validationid", "validid", "valindex"
)

def read_id_sets_from_path(path: Path) -> List[Tuple[str, set]]:
    """Extract plausible validation identifiers from a file."""
    results = []
    suffix = path.suffix.lower()

    try:
        if suffix == ".npy":
            arr = np.load(path, allow_pickle=True)
            results.append((path.name, set(map(str, np.asarray(arr).ravel().tolist()))))
        elif suffix == ".npz":
            data = np.load(path, allow_pickle=True)
            for key in data.files:
                if any(h in normalize_name(key) for h in ID_NAME_HINTS):
                    arr = np.asarray(data[key]).ravel()
                    results.append((f"{path.name}:{key}", set(map(str, arr.tolist()))))
        else:
            for i, df in enumerate(read_tabular_file(path)):
                for col in df.columns:
                    norm = normalize_name(col)
                    if norm in ID_COLUMN_HINTS or (
                        any(h in norm for h in ["imageid", "lesionid", "sampleid", "validx", "valindex"])
                    ):
                        vals = df[col].dropna().astype(str).str.strip()
                        vals = vals[vals != ""]
                        if len(vals) > 0:
                            results.append((f"{path.name}:sheet/frame{i}:{col}", set(vals.tolist())))
    except Exception:
        pass
    return [(label, ids) for label, ids in results if ids]

def discover_validation_id_sets(folder: Path) -> Dict[int, List[Tuple[str, set]]]:
    by_fold: Dict[int, List[Tuple[str, set]]] = defaultdict(list)
    for path in folder.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in ID_FILE_EXTENSIONS:
            continue
        lower = str(path).lower()
        if not any(h in lower for h in ID_NAME_HINTS):
            continue
        fold_raw = parse_fold_from_text(lower)
        if fold_raw is None:
            continue
        fold = fold_raw + 1 if fold_raw in range(0, 5) else fold_raw
        if fold not in range(1, 6):
            continue
        for label, ids in read_id_sets_from_path(path):
            by_fold[fold].append((str(path) + " | " + label, ids))
    return dict(by_fold)

def best_matching_id_pair(
    left_sets: List[Tuple[str, set]], right_sets: List[Tuple[str, set]]
) -> Optional[Dict[str, Any]]:
    if not left_sets or not right_sets:
        return None
    candidates = []
    for l_label, l_ids in left_sets:
        for r_label, r_ids in right_sets:
            union = l_ids | r_ids
            jaccard = len(l_ids & r_ids) / len(union) if union else 1.0
            candidates.append((jaccard, len(l_ids), len(r_ids), l_label, r_label, l_ids == r_ids))
    candidates.sort(reverse=True, key=lambda x: (x[0], min(x[1], x[2])))
    j, nl, nr, ll, rl, equal = candidates[0]
    return {
        "equal": equal, "jaccard": j, "left_n": nl, "right_n": nr,
        "left_source": ll, "right_source": rl
    }

id_inventory = {
    exp: discover_validation_id_sets(EXPERIMENT_DIRS[exp])
    for exp in [f"E{i}" for i in range(1, 11)]
}

split_verification_rows = []
for attention_exp, baseline_exp in COMPARISONS:
    for fold in range(1, 6):
        match = best_matching_id_pair(
            id_inventory[attention_exp].get(fold, []),
            id_inventory[baseline_exp].get(fold, [])
        )
        if match is None:
            split_verification_rows.append({
                "comparison": f"{attention_exp} vs {baseline_exp}",
                "fold": fold,
                "status": "ID information unavailable",
                "identical": np.nan,
                "jaccard": np.nan,
                "attention_id_count": np.nan,
                "baseline_id_count": np.nan,
                "attention_source": "",
                "baseline_source": "",
            })
        else:
            split_verification_rows.append({
                "comparison": f"{attention_exp} vs {baseline_exp}",
                "fold": fold,
                "status": "IDENTICAL" if match["equal"] else "MISMATCH",
                "identical": match["equal"],
                "jaccard": match["jaccard"],
                "attention_id_count": match["left_n"],
                "baseline_id_count": match["right_n"],
                "attention_source": match["left_source"],
                "baseline_source": match["right_source"],
            })

split_verification_df = pd.DataFrame(split_verification_rows)
display(split_verification_df)

mismatch_rows = split_verification_df[split_verification_df["status"] == "MISMATCH"]
if not mismatch_rows.empty:
    raise RuntimeError(
        "Validation split identifier mismatch detected. Paired statistical testing "
        "has been stopped.\n\n" + mismatch_rows.to_string(index=False)
    )

def pairing_statement_for_comparison(comparison: str) -> str:
    rows = split_verification_df[split_verification_df["comparison"] == comparison]
    if len(rows) == 5 and (rows["status"] == "IDENTICAL").all():
        return "Validation identifiers were available and identical for all five corresponding folds."
    return (
        "Validation identifiers were not available for every fold; pairing was therefore "
        "based on matching fold numbers and the shared splitter configuration within the protocol."
    )

print("\nPAIRING VERIFICATION SUMMARY")
for attention_exp, baseline_exp in COMPARISONS:
    comp = f"{attention_exp} vs {baseline_exp}"
    print(f"- {comp}: {pairing_statement_for_comparison(comp)}")


,comparison,fold,status,identical,jaccard,attention_id_count,baseline_id_count,attention_source,baseline_source
0,E2 vs E1,1,ID information unavailable,NaN,NaN,NaN,NaN,,
1,E2 vs E1,2,ID information unavailable,NaN,NaN,NaN,NaN,,
2,E2 vs E1,3,ID information unavailable,NaN,NaN,NaN,NaN,,
3,E2 vs E1,4,ID information unavailable,NaN,NaN,NaN,NaN,,
4,E2 vs E1,5,ID information unavailable,NaN,NaN,NaN,NaN,,
5,E3 vs E1,1,ID information unavailable,NaN,NaN,NaN,NaN,,
6,E3 vs E1,2,ID information unavailable,NaN,NaN,NaN,NaN,,
7,E3 vs E1,3,ID information unavailable,NaN,NaN,NaN,NaN,,
8,E3 vs E1,4,ID information unavailable,NaN,NaN,NaN,NaN,,
9,E3 vs E1,5,ID information unavailable,NaN,NaN,NaN,NaN,,



PAIRING VERIFICATION SUMMARY
- E2 vs E1: Validation identifiers were not available for every fold; pairing was therefore based on matching fold numbers and the shared splitter configuration within the protocol.
- E3 vs E1: Validation identifiers were not available for every fold; pairing was therefore based on matching fold numbers and the shared splitter configuration within the protocol.
- E4 vs E1: Validation identifiers were not available for every fold; pairing was therefore based on matching fold numbers and the shared splitter configuration within the protocol.
- E5 vs E1: Validation identifiers were not available for every fold; pairing was therefore based on matching fold numbers and the shared splitter configuration within the protocol.
- E7 vs E6: Validation identifiers were not available for every fold; pairing was therefore based on matching fold numbers and the shared splitter configuration within the protocol.
- E8 vs E6: Validation identifiers were not available for ev

In [7]:

# =========================
# 6. STATISTICAL FUNCTIONS
# =========================
def exact_two_sided_wilcoxon(differences: Sequence[float]) -> Tuple[float, float]:
    """
    Exact two-sided Wilcoxon signed-rank test by exhaustive sign enumeration.

    Zeros are removed (Wilcox convention). Average ranks are used for ties.
    The two-sided p-value is P(T <= observed T), where
    T = min(W+, total_rank - W+), under all 2^n sign assignments.
    This remains exact for the observed absolute ranks and is suitable here
    because n <= 5.
    """
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    d = d[d != 0]

    if len(d) == 0:
        return 0.0, 1.0

    ranks = stats.rankdata(np.abs(d), method="average")
    w_plus = float(ranks[d > 0].sum())
    total = float(ranks.sum())
    observed_t = min(w_plus, total - w_plus)

    enumerated_t = []
    for signs in product([0, 1], repeat=len(ranks)):
        wp = float(sum(rank for rank, positive in zip(ranks, signs) if positive))
        enumerated_t.append(min(wp, total - wp))

    p_value = float(np.mean(np.asarray(enumerated_t) <= observed_t + 1e-12))
    return observed_t, min(1.0, p_value)

def cohen_dz(differences: Sequence[float]) -> float:
    d = np.asarray(differences, dtype=float)
    sd = d.std(ddof=1)
    if np.isclose(sd, 0):
        if np.isclose(d.mean(), 0):
            return 0.0
        return math.copysign(np.inf, d.mean())
    return float(d.mean() / sd)

def effect_size_label(dz: float) -> str:
    a = abs(dz)
    if a < 0.20:
        return "negligible"
    if a < 0.50:
        return "small"
    if a < 0.80:
        return "moderate"
    return "large"

def format_p(p: float) -> str:
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def latex_escape(text: str) -> str:
    replacements = {
        "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
        "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}", "\\": r"\textbackslash{}",
    }
    return "".join(replacements.get(ch, ch) for ch in str(text))


In [8]:

# =========================
# 7. EIGHT PAIRED COMPARISONS
# =========================
comparison_rows = []
difference_vectors = {}

for attention_exp, baseline_exp in COMPARISONS:
    att = (
        fold_df[fold_df["experiment"] == attention_exp]
        .sort_values("fold")
        .set_index("fold")["macro_f1"]
    )
    base = (
        fold_df[fold_df["experiment"] == baseline_exp]
        .sort_values("fold")
        .set_index("fold")["macro_f1"]
    )

    if not att.index.equals(base.index) or att.index.tolist() != [1, 2, 3, 4, 5]:
        raise RuntimeError(
            f"{attention_exp} vs {baseline_exp}: fold indices are not correctly paired."
        )

    differences = (att - base).to_numpy(float)
    difference_vectors[f"{attention_exp} vs {baseline_exp}"] = differences

    n = len(differences)
    mean_diff = float(differences.mean())
    sd_diff = float(differences.std(ddof=1))
    se_diff = sd_diff / math.sqrt(n)
    t_critical = stats.t.ppf(0.975, df=n - 1)
    ci_lower = mean_diff - t_critical * se_diff
    ci_upper = mean_diff + t_critical * se_diff

    t_stat, p_t = stats.ttest_rel(att.to_numpy(float), base.to_numpy(float))
    w_stat, p_w = exact_two_sided_wilcoxon(differences)
    dz = cohen_dz(differences)

    comparison_rows.append({
        "protocol": EXPERIMENT_META[attention_exp]["protocol"],
        "comparison": f"{attention_exp} vs {baseline_exp}",
        "attention_experiment": attention_exp,
        "baseline_experiment": baseline_exp,
        "attention_mean": float(att.mean()),
        "baseline_mean": float(base.mean()),
        "mean_difference_pp": mean_diff,
        "sd_difference": sd_diff,
        "ci95_lower": float(ci_lower),
        "ci95_upper": float(ci_upper),
        "t_statistic": float(t_stat),
        "p_ttest_raw": float(p_t),
        "cohens_dz": float(dz),
        "wilcoxon_statistic": float(w_stat),
        "p_wilcoxon_raw": float(p_w),
        "positive_folds": int(np.sum(differences > 0)),
        "negative_folds": int(np.sum(differences < 0)),
        "zero_folds": int(np.sum(differences == 0)),
    })

stats_df = pd.DataFrame(comparison_rows)

stats_df["p_ttest_holm"] = multipletests(
    stats_df["p_ttest_raw"].to_numpy(), alpha=0.05, method="holm"
)[1]
stats_df["p_wilcoxon_holm"] = multipletests(
    stats_df["p_wilcoxon_raw"].to_numpy(), alpha=0.05, method="holm"
)[1]

def make_interpretation(row: pd.Series) -> str:
    direction = (
        "numerically higher" if row["mean_difference_pp"] > 0
        else "numerically lower" if row["mean_difference_pp"] < 0
        else "numerically equal"
    )
    corrected_sig = row["p_ttest_holm"] < 0.05
    es = effect_size_label(row["cohens_dz"])

    if corrected_sig:
        if row["mean_difference_pp"] > 0:
            conclusion = "higher after Holm correction"
        elif row["mean_difference_pp"] < 0:
            conclusion = "lower after Holm correction"
        else:
            conclusion = "different after Holm correction"
    else:
        conclusion = f"{direction}, but not statistically significant after Holm correction"

    return (
        f"The attention model was {conclusion}; paired effect size was {es} "
        f"(Cohen's dz={row['cohens_dz']:.2f})."
    )

stats_df["interpretation"] = stats_df.apply(make_interpretation, axis=1)

required_order = [
    "protocol", "comparison", "attention_experiment", "baseline_experiment",
    "attention_mean", "baseline_mean", "mean_difference_pp", "sd_difference",
    "ci95_lower", "ci95_upper", "t_statistic", "p_ttest_raw", "p_ttest_holm",
    "cohens_dz", "wilcoxon_statistic", "p_wilcoxon_raw", "p_wilcoxon_holm",
    "positive_folds", "negative_folds", "zero_folds", "interpretation"
]
stats_df = stats_df[required_order]

print("PAIRED DIFFERENCE VECTORS (ATTENTION − BASELINE), PERCENTAGE POINTS")
for comparison, vector in difference_vectors.items():
    print(f"{comparison}: {np.round(vector, 4).tolist()}")

print("\nCOMPLETE STATISTICAL TABLE")
display(stats_df.round({
    "attention_mean": 4, "baseline_mean": 4, "mean_difference_pp": 4,
    "sd_difference": 4, "ci95_lower": 4, "ci95_upper": 4,
    "t_statistic": 4, "p_ttest_raw": 6, "p_ttest_holm": 6,
    "cohens_dz": 4, "wilcoxon_statistic": 4,
    "p_wilcoxon_raw": 6, "p_wilcoxon_holm": 6,
}))


PAIRED DIFFERENCE VECTORS (ATTENTION − BASELINE), PERCENTAGE POINTS
E2 vs E1: [1.805, 1.0693, 0.386, 0.0971, 2.9313]
E3 vs E1: [0.2241, -0.4167, -2.7819, -1.6736, 2.8709]
E4 vs E1: [3.4147, -2.9023, 0.5448, -3.6293, -2.7169]
E5 vs E1: [-0.7139, 0.3147, -2.2963, -3.8025, -1.0996]
E7 vs E6: [-5.2665, -1.4943, 0.8542, 1.5769, -6.0099]
E8 vs E6: [-1.0811, 4.2546, 1.8756, 5.3039, -3.4654]
E9 vs E6: [-4.1655, 1.7389, -1.4594, 3.3057, -1.1652]
E10 vs E6: [-2.0075, -0.7387, 0.1312, 6.4678, -6.678]

COMPLETE STATISTICAL TABLE


,protocol,comparison,attention_experiment,baseline_experiment,attention_mean,baseline_mean,mean_difference_pp,sd_difference,ci95_lower,ci95_upper,...,p_ttest_raw,p_ttest_holm,cohens_dz,wilcoxon_statistic,p_wilcoxon_raw,p_wilcoxon_holm,positive_folds,negative_folds,zero_folds,interpretation
0,Image-level Stratified 5-Fold,E2 vs E1,E2,E1,84.6760,83.4182,1.2577,1.1449,-0.1638,2.6793,...,0.069953,0.559625,1.0986,0.0,0.0625,0.500,5,0,0,"The attention model was numerically higher, bu..."
1,Image-level Stratified 5-Fold,E3 vs E1,E3,E1,83.0628,83.4182,-0.3555,2.1433,-3.0167,2.3057,...,0.729551,1.000000,-0.1659,6.0,0.8125,1.000,2,3,0,"The attention model was numerically lower, but..."
2,Image-level Stratified 5-Fold,E4 vs E1,E4,E1,82.3604,83.4182,-1.0578,2.9723,-4.7484,2.6328,...,0.470716,1.000000,-0.3559,5.0,0.6250,1.000,2,3,0,"The attention model was numerically lower, but..."
3,Image-level Stratified 5-Fold,E5 vs E1,E5,E1,81.8987,83.4182,-1.5195,1.5816,-3.4833,0.4443,...,0.098166,0.687160,-0.9608,1.0,0.1250,0.875,1,4,0,"The attention model was numerically lower, but..."
4,Lesion-level Group 5-Fold,E7 vs E6,E7,E6,70.5132,72.5812,-2.0679,3.4613,-6.3657,2.2299,...,0.252522,1.000000,-0.5974,4.0,0.4375,1.000,2,3,0,"The attention model was numerically lower, but..."
5,Lesion-level Group 5-Fold,E8 vs E6,E8,E6,73.9587,72.5812,1.3775,3.6552,-3.1610,5.9160,...,0.446835,1.000000,0.3769,4.0,0.4375,1.000,3,2,0,"The attention model was numerically higher, bu..."
6,Lesion-level Group 5-Fold,E9 vs E6,E9,E6,72.2320,72.5812,-0.3491,2.9232,-3.9788,3.2806,...,0.802634,1.000000,-0.1194,7.0,1.0000,1.000,2,3,0,"The attention model was numerically lower, but..."
7,Lesion-level Group 5-Fold,E10 vs E6,E10,E6,72.0161,72.5812,-0.5650,4.7282,-6.4359,5.3058,...,0.802511,1.000000,-0.1195,5.0,0.6250,1.000,2,3,0,"The attention model was numerically lower, but..."


In [9]:

# =========================
# 8. CREATE ACADEMIC TEXT FROM ACTUAL RESULTS
# =========================
def result_sentence(row: pd.Series) -> str:
    attention = row["attention_experiment"]
    baseline = row["baseline_experiment"]
    direction = "higher" if row["mean_difference_pp"] > 0 else "lower" if row["mean_difference_pp"] < 0 else "equal"
    significance = (
        "This difference remained statistically significant after Holm correction"
        if row["p_ttest_holm"] < 0.05
        else "This difference was not statistically significant after Holm correction"
    )
    return (
        f"{attention} versus {baseline} was numerically {direction} by "
        f"{abs(row['mean_difference_pp']):.2f} percentage points "
        f"(95% CI {row['ci95_lower']:.2f} to {row['ci95_upper']:.2f}; "
        f"Cohen's dz={row['cohens_dz']:.2f}; raw p={format_p(row['p_ttest_raw'])}; "
        f"Holm-adjusted p={format_p(row['p_ttest_holm'])}). {significance}."
    )

all_ids_verified = (
    len(split_verification_df) == 40
    and (split_verification_df["status"] == "IDENTICAL").all()
)
pairing_clause = (
    "Pairing was verified using identical validation identifiers for every corresponding fold."
    if all_ids_verified else
    "Where explicit validation identifiers were unavailable, pairing was established by the corresponding fold number and the common splitter configuration used within each validation protocol."
)

methodology_text = (
    "Fold-level statistical comparisons were performed separately within the image-level "
    "Stratified five-fold and lesion-level Group five-fold validation protocols. For each "
    "attention variant, the macro-F1 score from a given fold was paired with the macro-F1 "
    "score from the same fold of the attention-free ConvNeXt-Tiny baseline. "
    f"{pairing_clause} The primary analysis used a two-sided paired-samples t-test. "
    "For each comparison, the paired mean difference (attention minus baseline) was reported "
    "in percentage points together with its 95% confidence interval and Cohen's dz paired "
    "effect size. An exact two-sided Wilcoxon signed-rank test was additionally conducted as "
    "a sensitivity analysis. Family-wise multiplicity across the eight baseline comparisons "
    "was controlled using the Holm procedure, applied separately to the paired t-test and "
    "Wilcoxon p-values; raw and adjusted p-values were retained. Because only five folds were "
    "available per experiment, these inferential analyses were considered exploratory, and "
    "effect-size magnitude was interpreted only as a descriptive guide rather than a substitute "
    "for statistical uncertainty."
)

e2 = stats_df.loc[stats_df["comparison"] == "E2 vs E1"].iloc[0]
e8 = stats_df.loc[stats_df["comparison"] == "E8 vs E6"].iloc[0]

strat_rows = stats_df[stats_df["protocol"].str.startswith("Image-level")]
group_rows = stats_df[stats_df["protocol"].str.startswith("Lesion-level")]
sig_rows = stats_df[stats_df["p_ttest_holm"] < 0.05]

strat_summary = "; ".join(result_sentence(r) for _, r in strat_rows.iterrows())
group_summary = "; ".join(result_sentence(r) for _, r in group_rows.iterrows())

if sig_rows.empty:
    multiplicity_conclusion = (
        "None of the eight paired t-test comparisons remained statistically significant after Holm correction."
    )
else:
    names = ", ".join(sig_rows["comparison"].tolist())
    multiplicity_conclusion = (
        f"The comparisons that remained statistically significant after Holm correction were: {names}."
    )

results_text = (
    "In the image-level Stratified validation protocol, Channel Attention (E2) produced the "
    f"highest mean macro-F1 and was {('numerically higher' if e2['mean_difference_pp'] > 0 else 'numerically lower')} "
    f"than the baseline E1 by {abs(e2['mean_difference_pp']):.2f} percentage points "
    f"(95% CI {e2['ci95_lower']:.2f} to {e2['ci95_upper']:.2f}; Cohen's dz={e2['cohens_dz']:.2f}; "
    f"raw p={format_p(e2['p_ttest_raw'])}; Holm-adjusted p={format_p(e2['p_ttest_holm'])}). "
    f"Across all image-level comparisons: {strat_summary} "
    "In the lesion-level Group validation protocol, Spatial Attention (E8) was the only "
    "attention mechanism with a mean macro-F1 above the baseline E6; its paired mean difference "
    f"was {e8['mean_difference_pp']:.2f} percentage points "
    f"(95% CI {e8['ci95_lower']:.2f} to {e8['ci95_upper']:.2f}; Cohen's dz={e8['cohens_dz']:.2f}; "
    f"raw p={format_p(e8['p_ttest_raw'])}; Holm-adjusted p={format_p(e8['p_ttest_holm'])}). "
    f"Across all lesion-level comparisons: {group_summary} {multiplicity_conclusion} "
    "Given the five-fold sample size, these findings should be interpreted as exploratory."
)

reviewer_text = (
    "Response: We thank the reviewer for requesting a fold-level assessment of statistical "
    "uncertainty. We have added eight paired comparisons between each attention model and its "
    "protocol-matched attention-free ConvNeXt-Tiny baseline. The five original macro-F1 values "
    "were paired by corresponding fold; no summarized mean or standard-deviation values were "
    "used as test inputs. The primary analysis was a two-sided paired-samples t-test, accompanied "
    "by the paired mean difference and 95% confidence interval in percentage points and Cohen's dz. "
    "We also added an exact two-sided Wilcoxon signed-rank sensitivity analysis. To account for "
    "the eight comparisons, Holm correction was applied separately to the paired t-test and "
    "Wilcoxon p-values, while both raw and adjusted p-values are reported. "
    f"{multiplicity_conclusion} Specifically, E2 versus E1 showed a paired mean difference of "
    f"{e2['mean_difference_pp']:.2f} percentage points (95% CI {e2['ci95_lower']:.2f} to "
    f"{e2['ci95_upper']:.2f}; Holm-adjusted p={format_p(e2['p_ttest_holm'])}), whereas E8 versus "
    f"E6—the only lesion-level attention variant with a higher mean macro-F1 than its baseline—"
    f"showed a paired mean difference of {e8['mean_difference_pp']:.2f} percentage points "
    f"(95% CI {e8['ci95_lower']:.2f} to {e8['ci95_upper']:.2f}; "
    f"Holm-adjusted p={format_p(e8['p_ttest_holm'])}). Because each comparison contains only five "
    "paired folds, we explicitly characterize the analysis as exploratory and avoid claims that "
    "extend beyond the corrected statistical evidence."
)

print("A. METHODOLOGY PARAGRAPH\n")
print(methodology_text)
print("\n\nB. RESULTS PARAGRAPH\n")
print(results_text)
print("\n\nC. RESPONSE TO REVIEWER\n")
print(reviewer_text)


A. METHODOLOGY PARAGRAPH

Fold-level statistical comparisons were performed separately within the image-level Stratified five-fold and lesion-level Group five-fold validation protocols. For each attention variant, the macro-F1 score from a given fold was paired with the macro-F1 score from the same fold of the attention-free ConvNeXt-Tiny baseline. Where explicit validation identifiers were unavailable, pairing was established by the corresponding fold number and the common splitter configuration used within each validation protocol. The primary analysis used a two-sided paired-samples t-test. For each comparison, the paired mean difference (attention minus baseline) was reported in percentage points together with its 95% confidence interval and Cohen's dz paired effect size. An exact two-sided Wilcoxon signed-rank test was additionally conducted as a sensitivity analysis. Family-wise multiplicity across the eight baseline comparisons was controlled using the Holm procedure, applied se

In [12]:

# =========================
# 9. WRITE CSV, LATEX, AND MARKDOWN OUTPUTS
# =========================
fold_output = fold_df[
    ["experiment", "protocol", "attention", "fold", "macro_f1"]
].copy()
fold_output.to_csv(OUTPUT_DIR / "fold_level_macro_f1.csv", index=False)

stats_df.to_csv(OUTPUT_DIR / "baseline_statistical_comparison.csv", index=False)

# Compact LaTeX table.
latex_lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Paired fold-level macro-F1 comparisons against the protocol-matched baseline. "
    r"Mean differences and confidence intervals are expressed in percentage points.}",
    r"\label{tab:fold_level_statistics}",
    r"\small",
    r"\begin{tabular}{llrrrrr}",
    r"\toprule",
    r"Protocol & Comparison & Mean difference (pp) & 95\% CI & Cohen's $d_z$ & Raw $p$ & Holm-adjusted $p$ \\",
    r"\midrule",
]

for _, row in stats_df.iterrows():
    protocol_short = (
        "Stratified" if row["protocol"].startswith("Image-level")
        else "Group"
    )
    ci = f"[{row['ci95_lower']:.2f}, {row['ci95_upper']:.2f}]"
    latex_lines.append(
        f"{latex_escape(protocol_short)} & {latex_escape(row['comparison'])} & "
        f"{row['mean_difference_pp']:.2f} & {ci} & {row['cohens_dz']:.2f} & "
        f"{format_p(row['p_ttest_raw'])} & {format_p(row['p_ttest_holm'])} \\\\"
    )

latex_lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\begin{flushleft}",
    r"\footnotesize Note: Two-sided paired-samples t-tests were used as the primary analysis. "
    r"Holm adjustment was applied across all eight paired t-tests. Because each comparison "
    r"contains five folds, the analysis is exploratory. Exact Wilcoxon sensitivity results "
    r"are provided in the supplementary statistical output.",
    r"\end{flushleft}",
    r"\end{table}",
]
latex_text = "\n".join(latex_lines)
(OUTPUT_DIR / "baseline_statistical_comparison.tex").write_text(
    latex_text, encoding="utf-8"
)

# Build detailed markdown report.
file_discovery_md = []
for exp in [f"E{i}" for i in range(1, 11)]:
    file_discovery_md.append(f"### {exp}")
    file_discovery_md.append(f"- Folder: `{discovery_log[exp]['folder']}`")
    file_discovery_md.append(f"- Extraction: {discovery_log[exp]['note']}")
    file_discovery_md.append("- Files containing selected macro-F1 candidates:")
    file_discovery_md.extend(f"  - `{p}`" for p in discovery_log[exp]["files"])

fold_table_md = (
    fold_output.pivot(index="experiment", columns="fold", values="macro_f1")
    .rename(columns=lambda x: f"Fold {x}")
    .round(4)
    .to_markdown()
)
quality_md = quality_df.round(4).to_markdown(index=False)
stats_md = stats_df.round({
    "attention_mean": 4, "baseline_mean": 4, "mean_difference_pp": 4,
    "sd_difference": 4, "ci95_lower": 4, "ci95_upper": 4,
    "t_statistic": 4, "p_ttest_raw": 6, "p_ttest_holm": 6,
    "cohens_dz": 4, "wilcoxon_statistic": 4,
    "p_wilcoxon_raw": 6, "p_wilcoxon_holm": 6,
}).to_markdown(index=False)

split_summary_lines = []
for attention_exp, baseline_exp in COMPARISONS:
    comp = f"{attention_exp} vs {baseline_exp}"
    split_summary_lines.append(f"- **{comp}:** {pairing_statement_for_comparison(comp)}")

report = f"""# Statistical Analysis Report

## Scope

This analysis used the original five fold-level macro-F1 values for ten completed HAM10000 experiments. No model training or re-training was performed. Macro-F1 values stored on a 0–1 scale were converted to percentages; values already stored on a 0–100 scale were retained.

## Files discovered

{chr(10).join(file_discovery_md)}

## Extracted fold-level macro-F1 values (%)

{fold_table_md}

## Data-quality and manuscript sanity checks

Each experiment was required to contain exactly five unique folds, no missing macro-F1 values, values within a plausible range, and an extracted mean and sample standard deviation consistent with the manuscript summary within the configured rounding tolerances.

{quality_md}

## Validation-split pairing verification

{chr(10).join(split_summary_lines)}

## Statistical methods

{methodology_text}

Cohen's dz magnitude was used only as a descriptive guide: |dz| < 0.20 negligible, 0.20–0.49 small, 0.50–0.79 moderate, and ≥ 0.80 large. Effect-size magnitude did not replace confidence intervals or multiplicity-adjusted inference.

## Complete paired-comparison results

{stats_md}

## Main results

{results_text}

## Limitations

Only five paired folds were available for each comparison. Consequently, estimates of standard deviation, confidence intervals, and p-values are imprecise and sensitive to individual folds. Cross-validation folds are also not equivalent to five independently sampled external datasets. The statistical analysis should therefore be interpreted as exploratory and as a quantification of fold-level consistency rather than definitive evidence of general superiority.

## Manuscript-ready text

### A. Methodology

{methodology_text}

### B. Results

{results_text}

### C. Response to Reviewer

{reviewer_text}
"""

(OUTPUT_DIR / "statistical_analysis_report.md").write_text(
    report, encoding="utf-8"
)

print("OUTPUT FILES")
print("=" * 80)
for filename in [
    "fold_level_macro_f1.csv",
    "baseline_statistical_comparison.csv",
    "baseline_statistical_comparison.tex",
    "statistical_analysis_report.md",
]:
    path = OUTPUT_DIR / filename
    print(f"{filename}: {path}")

print("\nAll requested outputs were written successfully.")


OUTPUT FILES
fold_level_macro_f1.csv: /Users/asfanme/RISET/Riset4/statistical_analysis_outputs/fold_level_macro_f1.csv
baseline_statistical_comparison.csv: /Users/asfanme/RISET/Riset4/statistical_analysis_outputs/baseline_statistical_comparison.csv
baseline_statistical_comparison.tex: /Users/asfanme/RISET/Riset4/statistical_analysis_outputs/baseline_statistical_comparison.tex
statistical_analysis_report.md: /Users/asfanme/RISET/Riset4/statistical_analysis_outputs/statistical_analysis_report.md

All requested outputs were written successfully.


In [11]:
%pip install tabulate

Note: you may need to restart the kernel to use updated packages.



## Notes on strict execution

- The notebook never inserts fold values manually.
- It stops before statistical testing when extraction or sanity checks fail.
- It stops when validation identifiers are available but reveal a mismatch.
- Exact Wilcoxon p-values are computed by exhaustive sign enumeration, which is feasible and transparent for five paired folds.
- The generated prose automatically follows the Holm-adjusted paired t-test results and uses “numerically higher/lower” when corrected significance is absent.
